# 🚀 SpaceX Falcon 9 Landing Prediction

Binary classification: predict whether the Falcon 9 first-stage booster lands successfully.

**IBM Data Science Professional Certificate — Applied Capstone**

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')
print("✅ Libraries loaded")

✅ Libraries loaded


## 2. Load Data

In [ ]:
# Try IBM dataset; fall back to synthetic data if offline
import urllib.request

IBM_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"

try:
    df = pd.read_csv(IBM_URL)
    print(f"✅ IBM dataset loaded: {df.shape}")
    FEATURES = [c for c in df.columns if c != 'Class']
    X = df[FEATURES].fillna(df[FEATURES].mean(numeric_only=True))
    Y = df['Class']
except Exception:
    print("⚠️  IBM URL unreachable — using synthetic SpaceX-like data")
    np.random.seed(42)
    n = 90
    df = pd.DataFrame({
        'PayloadMass':             np.random.uniform(500, 15000, n),
        'Flights':                 np.random.randint(1, 10, n),
        'Block':                   np.random.randint(1, 6, n),
        'ReusedCount':             np.random.randint(0, 5, n),
        'Orbit_LEO':               np.random.randint(0, 2, n),
        'Orbit_ISS':               np.random.randint(0, 2, n),
        'Orbit_GEO':               np.random.randint(0, 2, n),
        'Orbit_SSO':               np.random.randint(0, 2, n),
        'Orbit_VLEO':              np.random.randint(0, 2, n),
        'Orbit_PO':                np.random.randint(0, 2, n),
        'LaunchSite_KSC LC-39A':   np.random.randint(0, 2, n),
        'LaunchSite_CCAFS SLC-40': np.random.randint(0, 2, n),
        'LaunchSite_VAFB SLC-4E':  np.random.randint(0, 2, n),
        # Class: success more likely with more flights (reused boosters)
        'Class': (np.random.rand(n) < 0.5 + 0.04 * np.random.randint(0, 8, n)).astype(int),
    })
    FEATURES = [c for c in df.columns if c != 'Class']
    X = df[FEATURES]
    Y = df['Class']

print(f"Samples: {len(X)} | Features: {X.shape[1]} | Landing success rate: {Y.mean():.1%}")

⚠️  IBM URL unreachable — using synthetic SpaceX-like data
Samples: 90 | Features: 13 | Landing success rate: 60.0%


## 3. Train/Test Split & Scaling

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"✅ Train: {X_train_s.shape} | Test: {X_test_s.shape}")
print(f"   Test samples: {len(Y_test)}")

✅ Train: (72, 13) | Test: (18, 13)
   Test samples: 18


## 4. GridSearchCV — All Models

In [ ]:
# Logistic Regression
lr = GridSearchCV(LogisticRegression(max_iter=10000),
                  {'C':[0.01,0.1,1,10,100],'solver':['lbfgs']}, cv=10)
lr.fit(X_train_s, Y_train)
print(f"LR  best: {lr.best_params_}  CV={lr.best_score_:.3f}")

# SVM
svm = GridSearchCV(SVC(),
                   {'kernel':['linear','rbf','sigmoid'],
                    'C':[0.1,1,10],'gamma':['scale','auto']}, cv=10)
svm.fit(X_train_s, Y_train)
print(f"SVM best: {svm.best_params_}  CV={svm.best_score_:.3f}")

# Decision Tree
dt = GridSearchCV(DecisionTreeClassifier(),
                  {'criterion':['gini','entropy'],
                   'max_depth':[2,4,6,8],
                   'max_features':['sqrt','log2']}, cv=10)
dt.fit(X_train_s, Y_train)
print(f"DT  best: {dt.best_params_}  CV={dt.best_score_:.3f}")

# KNN
knn = GridSearchCV(KNeighborsClassifier(),
                   {'n_neighbors':[3,5,7,9,11],'p':[1,2]}, cv=10)
knn.fit(X_train_s, Y_train)
print(f"KNN best: {knn.best_params_}  CV={knn.best_score_:.3f}")

LR  best: {'C': 0.01, 'solver': 'lbfgs'}  CV=0.611
SVM best: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}  CV=0.639
DT  best: {'criterion': 'gini', 'max_depth': 2, 'max_features': 'sqrt'}  CV=0.625
KNN best: {'n_neighbors': 11, 'p': 1}  CV=0.611


## 5. Test Set Results

In [ ]:
print(f"{'Model':25s} {'CV Acc':>8} {'Test Acc':>10}")
print("-" * 47)
for name, model in [("Logistic Regression", lr), ("SVM", svm),
                     ("Decision Tree", dt),       ("KNN", knn)]:
    test_acc = accuracy_score(Y_test, model.predict(X_test_s))
    print(f"{name:25s} {model.best_score_:>8.3f} {test_acc:>10.3f}")

Model                       CV Acc   Test Acc
-----------------------------------------------
Logistic Regression         0.611      0.556
SVM                         0.639      0.556
Decision Tree               0.625      0.556
KNN                         0.611      0.556


## 6. Confusion Matrix

In [ ]:
best_model = svm   # change to lr/dt/knn to compare
y_pred = best_model.predict(X_test_s)
cm = confusion_matrix(Y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Failed','Landed'],
            yticklabels=['Failed','Landed'])
ax.set_title('Confusion Matrix — Best Model')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()

print(classification_report(Y_test, y_pred, target_names=['Failed','Landed']))

              precision    recall  f1-score   support

      Failed       0.50      0.57      0.53        7
      Landed       0.63      0.55      0.59       11

    accuracy                           0.56       18
   macro avg       0.57      0.56      0.56       18
weighted avg       0.58      0.56      0.57       18


## 7. Summary

| Model | GridSearchCV (CV) | Test Accuracy |
|---|---|---|
| Logistic Regression | see output | see output |
| SVM | see output | see output |
| Decision Tree | see output | see output |
| KNN | see output | see output |

> **Note:** Results depend on which dataset is loaded. With the real IBM SpaceX dataset (internet required), typical test accuracy is ~83% for LR/SVM/KNN.

**Completed as part of the IBM Data Science Professional Certificate (Coursera)**